# Encode Immune Human Dataset with scGPT

Encode the `Immune_ALL_human.h5ad` dataset using the pre-trained scGPT model.
For a specified cell type, compute average gene embeddings and write to a
`data/embeddings/<cell_type>.h5ad` file.

In [1]:
from pathlib import Path

import scanpy as sc
import torch

from src.scgpt import (
    create_scgpt_dataset,
    encode_scgpt_embeddings_to_h5ad,
    load_average_gene_embeddings,
    load_scgpt_model,
)

/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHT

## Configuration

In [ ]:
DATA_PATH = Path("../data/Immune_ALL_human.h5ad")
MODEL_DIR = Path("../models/scGPT_human")
# GENE_LIST_FILE = Path("../data/trrust_genes.txt")
OUTPUT_PATH = Path("../data/embeddings/scgpt_human_cd20.h5ad")
CELL_TYPE = "CD20+ B cells"
BATCH_SIZE = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Cell type: {CELL_TYPE}")
print(f"Output: {OUTPUT_PATH}")

Device: cuda
Cell type: CD20+ B cells
Output: ../data/embeddings/scgpt_human_cd20.h5ad


## Load model and data

In [3]:
bundle = load_scgpt_model(MODEL_DIR, device=DEVICE)
print(f"Model: {bundle.config['embsize']}d, {bundle.config['nlayers']} layers")
print(f"Vocab: {len(bundle.vocab)} genes")

Model: 512d, 12 layers
Vocab: 60697 genes


In [4]:
adata = sc.read(DATA_PATH, cache=True)
adata.obs["celltype"] = adata.obs["final_annotation"].astype(str)
print(f"Dataset: {adata.shape[0]} cells, {adata.shape[1]} genes")
print(f"Cell types: {adata.obs['celltype'].nunique()}")

Dataset: 33506 cells, 12303 genes
Cell types: 16


## Preprocess and encode

In [ ]:
adata_ct = adata[adata.obs["celltype"] == CELL_TYPE].copy()
ds = create_scgpt_dataset(
    adata_ct,
    vocab=bundle.vocab,
    gene2idx=bundle.gene2idx,
    # gene_list_file=GENE_LIST_FILE,
    batch_size=BATCH_SIZE,
)

print(f"Cells: {adata_ct.n_obs}")
print(f"Genes in vocab: {len(ds.genes_in_vocab)}")

encode_scgpt_embeddings_to_h5ad(
    model=bundle.model,
    dataloader=ds.dataloader,
    vocab=bundle.vocab,
    gene_names=ds.genes_in_vocab,
    cell_type=CELL_TYPE,
    output_path=OUTPUT_PATH,
    device=DEVICE,
)
print(f"Saved → {OUTPUT_PATH}")

Cells: 2873
scGPT - INFO - Filtering genes by counts ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Binning data ...
Genes in vocab: 10856


Encoding embeddings:   0%|          | 0/240 [00:00<?, ?it/s]/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)
Encoding embeddings: 100%|██████████| 240/240 [1:00:22<00:00, 15.09s/it]


Saved → ../data/embeddings/scgpt_human_cd20.h5ad


## Sanity check

In [6]:
result = load_average_gene_embeddings(OUTPUT_PATH)

print(f"Cell type: {result.uns['cell_type']!r}")
print(f"Genes: {result.n_obs}")
print(f"Embedding size: {result.X.shape[1]}")
print(f"Shape: {result.X.shape}")

Cell type: 'CD20+ B cells'
Genes: 10856
Embedding size: 512
Shape: (10856, 512)
